In [33]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from retrying import retry 

base_url = "https://www.bafin.de/SiteGlobals/Forms/Suche/EN/Expertensuche_Formular.html?gts=dateOfIssue_dt+desc&cl2Categories_Aufsichtsbereich=Wertpapieraufsicht&sortOrder=dateOfIssue_dt+desc"

HEADERS = {
    'sec-ch-ua': '"Avast Secure Browser";v="117", "Not;A=Brand";v="8", "Chromium";v="117"',
    'Referer': 'https://www.bafin.de/SiteGlobals/Forms/Suche/EN/Expertensuche_Formular.html?gts=dateOfIssue_dt+desc&cl2Categories_Aufsichtsbereich=Wertpapieraufsicht&sortOrder=dateOfIssue_dt+desc',
    'DNT': '1',
    'sec-ch-ua-mobile': '?0',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36 Avast/117.0.0.0',
    'sec-ch-ua-platform': '"Windows"',
}

current_page = 1
max_pages = 359

Titles = []
Dates = []
Desc = []
Title_Links = []
Attachment_Links = []
Topics = []

# Define a retry decorator
@retry(wait_fixed=2000, stop_max_attempt_number=3)
def get_url(url):
    response = requests.get(url, headers=HEADERS, timeout=(10, 30))
    response.raise_for_status()
    return response

while current_page <= max_pages:
    url = f"{base_url}&gtp=19572922_list%3D{current_page}"
    
    try:
        response = get_url(url)
    except Exception as e:
        print(f"Failed to fetch data for page {current_page}: {e}")
        continue  # Skip to the next page and retry
    
    soup = BeautifulSoup(response.content, 'html.parser')
    Content = soup.find_all('div', class_='search-result')

    for content in Content:
        title = content.select_one('h3 > a')
        date = content.find('span', class_='metadata').get_text()
        desc = content.find('p').get_text() if content.find('p') else ''
        attach = content.select_one('ul.links a').get('href') if content.select_one('ul.links a') else ''
        topic = content.select_one('span.thema').get_text() if content.select_one('span.thema') else ''
        Titles.append(title.get_text())
        Dates.append(date)
        Desc.append(desc)
        Title_Links.append('https://www.bafin.de/' + str(title.get('href')))
        Attachment_Links.append('https://www.bafin.de' + str(attach))
        Topics.append(topic)

    current_page += 1

data = {
    'Title': Titles,
    'Date': Dates,
    'Topic': Topics,
    'Description': Desc,
    'Title_Link': Title_Links,
    'Attachment_Link': Attachment_Links,
}

df = pd.DataFrame(data)
df


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('Bafin_raw_Data.csv')
df

,Title,Date,Topic,Description,Title_Link,Attachment_Link
0,TC Un­ter­hal­tungse­lek­tron­ik Ak­tienge­sel...,Format Measure Erscheinung: from 20.10.2023,Topic Measures,"On 10 October 2023, the Federal Financial Supe...",https://www.bafin.de/SharedDocs/Veroeffentlich...,https://www.bafin.de
1,TC Un­ter­hal­tungse­lek­tron­ik Ak­tienge­sel...,Format Measure Erscheinung: from 20.10.2023,Topic Measures,Die Finanzaufsicht BaFin hat am 10. Oktober 20...,https://www.bafin.de/SharedDocs/Veroeffentlich...,https://www.bafin.de
2,IT-Ad­min­is­tra­tor*in­nen (w/m/d) für den IT...,Format Attachment Erscheinung: from 20.10.2023,Topic Working at BaFin,Sachbearbeiter*in für eine unbefristete Tätigk...,https://www.bafin.de/SharedDocs/Downloads/DE/S...,https://www.bafin.de/SharedDocs/Downloads/DE/S...
3,BaFin er­mit­telt gegen die Sky­line Dig­i­tal...,Format News Erscheinung: from 19.10.2023,"Topic Unauthorised business, Consumer protection",Die Finanzaufsicht BaFin warnt vor Angeboten d...,https://www.bafin.de/SharedDocs/Veroeffentlich...,https://www.bafin.de
4,Prospekt fehlt: Di­a­gen­ics Group SE darf ihr...,Format Measure Erscheinung: from 19.10.2023,"Topic Measures, Prospectuses, Consumer protec...",Die Finanzaufsicht BaFin hat der Diagenics Gro...,https://www.bafin.de/SharedDocs/Veroeffentlich...,https://www.bafin.de
...,...,...,...,...,...,...
7163,Kryp­tover­wahrgeschäft,"Format Article Stand: , updated on 01.09.2022",Topic Fintechs,Das Kryptoverwahrgeschäft ist eine Finanzdiens...,https://www.bafin.de/DE/Aufsicht/FinTech/Gesch...,https://www.bafin.de
7164,"Di­en­stleis­tun­gen rund um DLT, Blockchain u...","Format Article Stand: , updated on 01.09.2022",Topic Fintechs,Innovative Finanztechnologien wie die Distribu...,https://www.bafin.de/DE/Aufsicht/FinTech/Gesch...,https://www.bafin.de
7165,Di­en­st­sitze und Wegbeschrei­bun­gen,Format Article,Topic BaFin,Die Dienstsitze der BaFin liegen in Bonn und F...,https://www.bafin.de/DE/DieBaFin/Kontakt/Wegbe...,https://www.bafin.de
7166,Was macht die BaFin nicht?,"Format Article Stand: , updated on 04.05.2020",Topic Consumer protection,Die BaFin beaufsichtigt Banken und Finanzdiens...,https://www.bafin.de/DE/Verbraucher/BaFinVerbr...,https://www.bafin.de


In [4]:
# Replace "https://www.bafin.de" with an empty string in the "Attachment_Link" column
df['Attachment_Link'] = df['Attachment_Link'].str.replace(r'^https://www\.bafin\.de$', '', regex=True)
df

,Title,Date,Topic,Description,Title_Link,Attachment_Link
0,TC Un­ter­hal­tungse­lek­tron­ik Ak­tienge­sel...,Format Measure Erscheinung: from 20.10.2023,Topic Measures,"On 10 October 2023, the Federal Financial Supe...",https://www.bafin.de/SharedDocs/Veroeffentlich...,
1,TC Un­ter­hal­tungse­lek­tron­ik Ak­tienge­sel...,Format Measure Erscheinung: from 20.10.2023,Topic Measures,Die Finanzaufsicht BaFin hat am 10. Oktober 20...,https://www.bafin.de/SharedDocs/Veroeffentlich...,
2,IT-Ad­min­is­tra­tor*in­nen (w/m/d) für den IT...,Format Attachment Erscheinung: from 20.10.2023,Topic Working at BaFin,Sachbearbeiter*in für eine unbefristete Tätigk...,https://www.bafin.de/SharedDocs/Downloads/DE/S...,https://www.bafin.de/SharedDocs/Downloads/DE/S...
3,BaFin er­mit­telt gegen die Sky­line Dig­i­tal...,Format News Erscheinung: from 19.10.2023,"Topic Unauthorised business, Consumer protection",Die Finanzaufsicht BaFin warnt vor Angeboten d...,https://www.bafin.de/SharedDocs/Veroeffentlich...,
4,Prospekt fehlt: Di­a­gen­ics Group SE darf ihr...,Format Measure Erscheinung: from 19.10.2023,"Topic Measures, Prospectuses, Consumer protec...",Die Finanzaufsicht BaFin hat der Diagenics Gro...,https://www.bafin.de/SharedDocs/Veroeffentlich...,
...,...,...,...,...,...,...
7163,Kryp­tover­wahrgeschäft,"Format Article Stand: , updated on 01.09.2022",Topic Fintechs,Das Kryptoverwahrgeschäft ist eine Finanzdiens...,https://www.bafin.de/DE/Aufsicht/FinTech/Gesch...,
7164,"Di­en­stleis­tun­gen rund um DLT, Blockchain u...","Format Article Stand: , updated on 01.09.2022",Topic Fintechs,Innovative Finanztechnologien wie die Distribu...,https://www.bafin.de/DE/Aufsicht/FinTech/Gesch...,
7165,Di­en­st­sitze und Wegbeschrei­bun­gen,Format Article,Topic BaFin,Die Dienstsitze der BaFin liegen in Bonn und F...,https://www.bafin.de/DE/DieBaFin/Kontakt/Wegbe...,
7166,Was macht die BaFin nicht?,"Format Article Stand: , updated on 04.05.2020",Topic Consumer protection,Die BaFin beaufsichtigt Banken und Finanzdiens...,https://www.bafin.de/DE/Verbraucher/BaFinVerbr...,


In [29]:
df['Date'] = df['Date'].str.replace('Erscheinung: from','')
df

,Title,Date,Topic,Description,Title_Link,Attachment_Link
0,TC Un­ter­hal­tungse­lek­tron­ik Ak­tienge­sel...,20.10.2023,Measures,"On 10 October 2023, the Federal Financial Supe...",https://www.bafin.de/SharedDocs/Veroeffentlich...,
1,TC Un­ter­hal­tungse­lek­tron­ik Ak­tienge­sel...,20.10.2023,Measures,Die Finanzaufsicht BaFin hat am 10. Oktober 20...,https://www.bafin.de/SharedDocs/Veroeffentlich...,
2,IT-Ad­min­is­tra­tor*in­nen (w/m/d) für den IT...,20.10.2023,Working at BaFin,Sachbearbeiter*in für eine unbefristete Tätigk...,https://www.bafin.de/SharedDocs/Downloads/DE/S...,https://www.bafin.de/SharedDocs/Downloads/DE/S...
3,BaFin er­mit­telt gegen die Sky­line Dig­i­tal...,19.10.2023,"Unauthorised business, Consumer protection",Die Finanzaufsicht BaFin warnt vor Angeboten d...,https://www.bafin.de/SharedDocs/Veroeffentlich...,
4,Prospekt fehlt: Di­a­gen­ics Group SE darf ihr...,19.10.2023,"Measures, Prospectuses, Consumer protection",Die Finanzaufsicht BaFin hat der Diagenics Gro...,https://www.bafin.de/SharedDocs/Veroeffentlich...,
...,...,...,...,...,...,...
7163,Kryp­tover­wahrgeschäft,01.09.2022,Fintechs,Das Kryptoverwahrgeschäft ist eine Finanzdiens...,https://www.bafin.de/DE/Aufsicht/FinTech/Gesch...,
7164,"Di­en­stleis­tun­gen rund um DLT, Blockchain u...",01.09.2022,Fintechs,Innovative Finanztechnologien wie die Distribu...,https://www.bafin.de/DE/Aufsicht/FinTech/Gesch...,
7165,Di­en­st­sitze und Wegbeschrei­bun­gen,,BaFin,Die Dienstsitze der BaFin liegen in Bonn und F...,https://www.bafin.de/DE/DieBaFin/Kontakt/Wegbe...,
7166,Was macht die BaFin nicht?,04.05.2020,Consumer protection,Die BaFin beaufsichtigt Banken und Finanzdiens...,https://www.bafin.de/DE/Verbraucher/BaFinVerbr...,


In [30]:
df.to_csv('BaFin_Data.csv', index=False)